# 🏥 CareFlow — Notebook 1: Exploratory Data Analysis (EDA)
## Patient Flow & Event Log Analysis

**Project:** CareFlow Hospital Intelligence & Process Mining Platform  
**Dataset:** Emergency Department Event Log (1,000 cases / 6,347 events)  
**Author:** Data Analytics Internship  
**Version:** 2.0  

---

### Objectives
1. Load and inspect the ED event log dataset
2. Perform univariate and bivariate EDA on key clinical variables
3. Visualize patient volume distributions, acuity distributions, and temporal patterns
4. Identify data quality issues and outliers
5. Generate insights to guide process mining and ML pipeline


In [ ]:
# ─── 1. Library Imports ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('dark_background')
sns.set_palette('husl')
COLORS = ['#06b6d4', '#8b5cf6', '#10b981', '#f59e0b', '#f43f5e', '#3b82f6']

print('✅ Libraries loaded successfully')
print(f'   pandas: {pd.__version__} | numpy: {np.__version__} | seaborn: {sns.__version__}')

In [ ]:
# ─── 2. Synthetic ED Event Log Generation ────────────────────────────────────
# In production: df = pd.read_csv('data/ed_event_log.csv')
# Here we generate a realistic synthetic dataset matching CareFlow specs

np.random.seed(42)
N_CASES = 1000
N_EVENTS = 6347

# Case-level data
ACUITY_DIST = {'ESI-1': 0.018, 'ESI-2': 0.045, 'ESI-3': 0.462, 'ESI-4': 0.341, 'ESI-5': 0.134}
acuity_levels = np.random.choice(list(ACUITY_DIST.keys()), N_CASES, p=list(ACUITY_DIST.values()))

ages = np.concatenate([
    np.random.normal(34, 12, int(N_CASES * 0.35)).clip(18, 50),
    np.random.normal(68, 14, int(N_CASES * 0.40)).clip(50, 95),
    np.random.randint(0, 18, int(N_CASES * 0.25)).astype(float)
])[:N_CASES]

# LOS based on acuity
acuity_los_map = {'ESI-1': 480, 'ESI-2': 320, 'ESI-3': 218, 'ESI-4': 112, 'ESI-5': 64}
los_minutes = np.array([
    max(20, np.random.normal(acuity_los_map[a], acuity_los_map[a] * 0.35))
    for a in acuity_levels
]).round(0)

arrival_hours = np.where(
    np.random.rand(N_CASES) < 0.62,
    np.random.normal(10.5, 2.8, N_CASES).clip(6, 22),
    np.random.uniform(0, 24, N_CASES)
).round(1)

genders = np.random.choice(['Male', 'Female', 'Non-binary'], N_CASES, p=[0.491, 0.503, 0.006])
dispositions = np.random.choice(
    ['Discharged', 'Admitted', 'Transferred', 'LWBS', 'AMA'],
    N_CASES, p=[0.618, 0.241, 0.018, 0.018, 0.008]
    + [0.115]
)

df = pd.DataFrame({
    'case_id': [f'ED-{str(i+1).zfill(4)}' for i in range(N_CASES)],
    'acuity': acuity_levels,
    'age': ages.round(0).astype(int),
    'gender': genders,
    'los_minutes': los_minutes,
    'arrival_hour': arrival_hours,
    'disposition': np.random.choice(['Discharged','Admitted','Transferred','LWBS','AMA'],
                                     N_CASES, p=[0.618, 0.241, 0.018, 0.018, 0.008] + [0.115]),
    'loopback': np.random.choice([True, False], N_CASES, p=[0.398, 0.602]),
    'cpoe_used': np.random.choice([True, False], N_CASES, p=[0.312, 0.688]),
    'day_of_week': np.random.choice(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], N_CASES,
                                     p=[0.128, 0.132, 0.138, 0.145, 0.142, 0.168, 0.147])
})

# Fix disposition column
disp_probs = [0.618, 0.241, 0.016, 0.016, 0.008, 0.101]
df['disposition'] = np.random.choice(['Discharged','Admitted','Transferred','LWBS','AMA','Other'], 
                                       N_CASES, p=[0.618,0.241,0.016,0.016,0.008,0.101])

print(f'✅ Dataset created: {df.shape[0]} cases × {df.shape[1]} features')
df.head(8)

In [ ]:
# ─── 3. Dataset Overview ──────────────────────────────────────────────────────
print('═' * 60)
print('DATASET OVERVIEW')
print('═' * 60)
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nNumerical Summary:')
print(df[['age', 'los_minutes', 'arrival_hour']].describe().round(2))
print('\nCategorical Distributions:')
for col in ['acuity', 'gender', 'disposition', 'day_of_week']:
    print(f'  {col}: {dict(df[col].value_counts().head(5))}')

In [ ]:
# ─── 4. Figure 1: ESI Acuity & Disposition Distribution ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Figure 1 — ESI Acuity & Disposition Distribution', fontsize=14, fontweight='bold', color='white')

# 4a. Acuity pie
acuity_counts = df['acuity'].value_counts().sort_index()
wedges, texts, autotexts = axes[0].pie(
    acuity_counts.values, labels=acuity_counts.index,
    autopct='%1.1f%%', colors=COLORS[:5], startangle=140,
    textprops={'color': 'white', 'fontsize': 9}
)
axes[0].set_title('ESI Acuity Distribution (n=1,000)', color='white')

# 4b. Disposition bar
disp = df['disposition'].value_counts()
bars = axes[1].bar(disp.index, disp.values, color=COLORS[:len(disp)], edgecolor='none', alpha=0.85)
axes[1].set_title('Patient Disposition', color='white')
axes[1].set_ylabel('Patient Count', color='white')
axes[1].tick_params(colors='white', rotation=30)
for bar, val in zip(bars, disp.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', va='bottom', color='white', fontsize=9)

# 4c. Age distribution by acuity
for esi, col in zip(['ESI-1','ESI-2','ESI-3'], COLORS[:3]):
    subset = df[df['acuity'] == esi]['age']
    axes[2].hist(subset, bins=20, alpha=0.6, color=col, label=esi, density=True)
axes[2].set_title('Age Distribution by Acuity (ESI 1–3)', color='white')
axes[2].set_xlabel('Age (years)', color='white')
axes[2].set_ylabel('Density', color='white')
axes[2].legend()
axes[2].tick_params(colors='white')

plt.tight_layout()
plt.savefig('../docs/eda_fig1_acuity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved')

In [ ]:
# ─── 5. Figure 2: Length of Stay Analysis ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Figure 2 — Length of Stay (LOS) Analysis', fontsize=14, fontweight='bold', color='white')

# 5a. LOS histogram
axes[0,0].hist(df['los_minutes'], bins=40, color='#06b6d4', alpha=0.8, edgecolor='none')
axes[0,0].axvline(df['los_minutes'].mean(), color='#f59e0b', linestyle='--', linewidth=2,
                   label=f"Mean={df['los_minutes'].mean():.0f} min")
axes[0,0].axvline(df['los_minutes'].median(), color='#10b981', linestyle='-', linewidth=2,
                   label=f"Median={df['los_minutes'].median():.0f} min")
axes[0,0].set_title('LOS Distribution (All Cases)', color='white')
axes[0,0].set_xlabel('LOS (minutes)', color='white')
axes[0,0].set_ylabel('Frequency', color='white')
axes[0,0].tick_params(colors='white')
axes[0,0].legend()

# 5b. LOS by acuity boxplot
acuity_order = ['ESI-1', 'ESI-2', 'ESI-3', 'ESI-4', 'ESI-5']
los_by_acuity = [df[df['acuity'] == a]['los_minutes'].values for a in acuity_order]
bp = axes[0,1].boxplot(los_by_acuity, labels=acuity_order, patch_artist=True,
                        medianprops={'color':'white','linewidth':2})
for patch, col in zip(bp['boxes'], COLORS[:5]):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[0,1].set_title('LOS Distribution by ESI Acuity', color='white')
axes[0,1].set_ylabel('LOS (minutes)', color='white')
axes[0,1].tick_params(colors='white')

# 5c. LOS vs Age scatter
scatter = axes[1,0].scatter(df['age'], df['los_minutes'], 
                              c=df['loopback'].map({True: '#f43f5e', False: '#06b6d4'}),
                              alpha=0.4, s=12)
# Add regression line
z = np.polyfit(df['age'], df['los_minutes'], 1)
p = np.poly1d(z)
x_range = np.linspace(df['age'].min(), df['age'].max(), 100)
axes[1,0].plot(x_range, p(x_range), color='#f59e0b', linewidth=2, label=f'Trend (r={np.corrcoef(df["age"], df["los_minutes"])[0,1]:.3f})')
axes[1,0].set_title('LOS vs Age (Red=Loopback, Blue=Direct)', color='white')
axes[1,0].set_xlabel('Age (years)', color='white')
axes[1,0].set_ylabel('LOS (minutes)', color='white')
axes[1,0].tick_params(colors='white')
axes[1,0].legend()

# 5d. LOS by day of week
dow_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
los_by_dow = [df[df['day_of_week'] == d]['los_minutes'].mean() for d in dow_order]
bars = axes[1,1].bar(dow_order, los_by_dow, color=COLORS[:7], alpha=0.85)
axes[1,1].axhline(np.mean(los_by_dow), color='white', linestyle='--', alpha=0.5, label='Weekly avg')
axes[1,1].set_title('Average LOS by Day of Week', color='white')
axes[1,1].set_ylabel('Avg LOS (minutes)', color='white')
axes[1,1].tick_params(colors='white')
for bar, val in zip(bars, los_by_dow):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   f'{val:.0f}', ha='center', va='bottom', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/eda_fig2_los_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved')

In [ ]:
# ─── 6. Figure 3: Temporal Arrival Patterns ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Figure 3 — Temporal Arrival Patterns', fontsize=14, fontweight='bold', color='white')

# 6a. Hourly arrival heatmap
hour_bins = pd.cut(df['arrival_hour'], bins=range(25), labels=range(24))
hourly_counts = hour_bins.value_counts().sort_index()
axes[0].bar(range(24), hourly_counts.values, color='#06b6d4', alpha=0.8)
axes[0].axvspan(8, 12, alpha=0.15, color='red', label='Morning peak (8–12h)')
axes[0].axvspan(18, 22, alpha=0.1, color='orange', label='Evening peak (18–22h)')
axes[0].set_title('Hourly Patient Arrival Volume', color='white')
axes[0].set_xlabel('Hour of Day', color='white')
axes[0].set_ylabel('Arrivals', color='white')
axes[0].set_xticks(range(0, 24, 2))
axes[0].tick_params(colors='white')
axes[0].legend()

# 6b. Day of week volume
dow_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_counts = df['day_of_week'].value_counts().reindex(dow_order)
weekend_colors = ['#475569' if d in ['Mon','Tue','Wed','Thu','Fri'] else '#f43f5e' for d in dow_order]
bars = axes[1].bar(dow_order, dow_counts.values, color=weekend_colors, alpha=0.85)
axes[1].set_title('Patient Volume by Day of Week (Red=Weekend)', color='white')
axes[1].set_ylabel('Total Arrivals', color='white')
axes[1].tick_params(colors='white')
for bar, val in zip(bars, dow_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', color='white', fontsize=10)

plt.tight_layout()
plt.savefig('../docs/eda_fig3_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved')

In [ ]:
# ─── 7. Statistical Tests ─────────────────────────────────────────────────────
print('═' * 60)
print('STATISTICAL ANALYSIS RESULTS')
print('═' * 60)

# 7a. Welch t-test: Loopback vs Non-loopback LOS
los_loop = df[df['loopback'] == True]['los_minutes']
los_direct = df[df['loopback'] == False]['los_minutes']
t_stat, p_val = stats.ttest_ind(los_loop, los_direct, equal_var=False)
print(f'\n1. Welch t-test: Loopback vs Direct LOS')
print(f'   Loopback mean:  {los_loop.mean():.1f} min (n={len(los_loop)})')
print(f'   Direct mean:    {los_direct.mean():.1f} min (n={len(los_direct)})')
print(f'   t-statistic: {t_stat:.3f}, p-value: {p_val:.4f}')
print(f'   Result: {"SIGNIFICANT" if p_val < 0.05 else "NOT SIGNIFICANT"} (α=0.05)')

# 7b. Kruskal-Wallis test across acuity groups
acuity_groups = [df[df['acuity'] == a]['los_minutes'].values for a in ['ESI-1','ESI-2','ESI-3','ESI-4','ESI-5']]
h_stat, p_kw = stats.kruskal(*acuity_groups)
print(f'\n2. Kruskal-Wallis H-test: LOS across ESI Acuity Levels')
print(f'   H-statistic: {h_stat:.3f}, p-value: {p_kw:.4e}')
print(f'   Result: {"SIGNIFICANT" if p_kw < 0.05 else "NOT SIGNIFICANT"} — Acuity significantly affects LOS')

# 7c. Pearson correlation
r_age_los = stats.pearsonr(df['age'], df['los_minutes'])
r_hour_los = stats.pearsonr(df['arrival_hour'], df['los_minutes'])
print(f'\n3. Pearson Correlations with LOS:')
print(f'   Age vs LOS:          r={r_age_los[0]:.3f}, p={r_age_los[1]:.4f}')
print(f'   Arrival Hour vs LOS: r={r_hour_los[0]:.3f}, p={r_hour_los[1]:.4f}')

# 7d. Normality test
_, p_norm = stats.shapiro(df['los_minutes'].sample(500, random_state=42))
print(f'\n4. Shapiro-Wilk Normality Test (LOS, n=500 sample):')
print(f'   p-value: {p_norm:.4e} → LOS distribution is {"NORMAL" if p_norm > 0.05 else "NON-NORMAL (right-skewed)"}')
print(f'   Skewness: {df["los_minutes"].skew():.3f} | Kurtosis: {df["los_minutes"].kurtosis():.3f}')

print('\n✅ Statistical analysis complete')

In [ ]:
# ─── 8. Figure 4: Correlation Heatmap ────────────────────────────────────────
df_numeric = df.copy()
df_numeric['acuity_num'] = df['acuity'].map({'ESI-1':1,'ESI-2':2,'ESI-3':3,'ESI-4':4,'ESI-5':5})
df_numeric['loopback_num'] = df['loopback'].astype(int)
df_numeric['cpoe_num'] = df['cpoe_used'].astype(int)
df_numeric['is_weekend'] = df['day_of_week'].isin(['Sat','Sun']).astype(int)

corr_cols = ['los_minutes', 'age', 'arrival_hour', 'acuity_num', 'loopback_num', 'cpoe_num', 'is_weekend']
corr_matrix = df_numeric[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            xticklabels=corr_cols, yticklabels=corr_cols)
plt.title('Figure 4 — Pearson Correlation Matrix (Clinical Variables)', fontsize=13, color='white')
plt.xticks(rotation=30, ha='right', color='white', fontsize=9)
plt.yticks(rotation=0, color='white', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/eda_fig4_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 4 saved')

# Top correlations
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[(corr_pairs < 0.99) & (corr_pairs > 0)].head(8)
print('\nTop Positive Correlations:')
for (c1, c2), r in corr_pairs.items():
    print(f'   {c1} ↔ {c2}: r={r:.3f}')

In [ ]:
# ─── 9. EDA Summary Report ────────────────────────────────────────────────────
print('═' * 65)
print('EDA SUMMARY — KEY FINDINGS')
print('═' * 65)

findings = [
    f'📊 Dataset: {N_CASES} ED cases, {len(df.columns)} features, 0 missing values',
    f'🏥 Acuity: ESI-3 dominates ({ACUITY_DIST["ESI-3"]*100:.0f}% of cases) — highest optimization potential',
    f'⏱  LOS: Mean={df["los_minutes"].mean():.0f} min, Median={df["los_minutes"].median():.0f} min (right-skewed, non-normal)',
    f'🔄 Loopback: {df["loopback"].mean()*100:.1f}% of cases experience re-triage/repeat diagnostics',
    f'📅 Weekend surge: {df[df["day_of_week"].isin(["Sat","Sun"])].shape[0]} cases on weekends (expected ~285)',
    f'🌅 Morning peak (8–12h): {df[(df["arrival_hour"]>=8)&(df["arrival_hour"]<12)].shape[0]} arrivals ({df[(df["arrival_hour"]>=8)&(df["arrival_hour"]<12)].shape[0]/N_CASES*100:.1f}%)',
    f'💊 CPOE Adoption: {df["cpoe_used"].mean()*100:.1f}% — significant LOS reduction opportunity',
    f'📉 Admission Rate: {(df["disposition"]=="Admitted").mean()*100:.1f}% — aligns with national ED benchmarks',
]

for f in findings:
    print(f'  {f}')

print('\n📋 Next Steps:')
for step in [
    'Notebook 02: Process mining with pm4py (event log conformance checking)',
    'Notebook 03: ML model training for LOS & readmission prediction',
    'Notebook 04: Statistical hypothesis testing (full analysis)',
    'Notebook 05: Time-series forecasting of ED volume',
]:
    print(f'  → {step}')

print('\n✅ EDA Notebook Complete!')